<a href="https://colab.research.google.com/github/Aman-8343/Text_Summarizer/blob/main/Text_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer,TrainingArguments,T5ForConditionalGeneration

In [5]:
!git clone https://github.com/Aman-8343/Text_Summarizer.git

Cloning into 'Text_Summarizer'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 26 (delta 8), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 3.83 MiB | 35.99 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [6]:
!ls Text_Summarizer/

anaconda_projects  samsum-train.csv	  Text_summarizer.ipynb
samsum-test.csv    samsum-validation.csv


In [7]:
import pandas as pd

train_data = pd.read_csv("Text_Summarizer/samsum-train.csv")
val_data = pd.read_csv("Text_Summarizer/samsum-validation.csv")
test_data = pd.read_csv("Text_Summarizer/samsum-test.csv")

In [8]:
#train_data=pd.read_csv("samsum-train.csv")
#val_data=pd.read_csv("samsum-validation.csv")

In [9]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."


In [10]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.shape

In [11]:
val_data.shape

(818, 3)

In [12]:
#random sampling
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [13]:
train_data.shape

(4000, 3)

# Pre Processing

In [14]:
import re

def clean_data(text):
    text=re.sub(r"\r\n"," ",text) #lines
    text=re.sub(r"\s+"," ",text) #spaces
    text=re.sub(r"<.*?"," ",text) #html tag
    text=text.strip().lower()
    return text

In [15]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [16]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:  file_other> claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenize

In [17]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [18]:
# raw data -> tokenized input for fine tuning

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length", max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length", max_length=128,truncation=True)

    inputs["labels"]=targets["input_ids"]
    return inputs

In [20]:
training_datasets=train_data.apply(tokenize,axis=1).tolist()
val_datasets=val_data.apply(tokenize,axis=1).tolist()

In [21]:
training_datasets[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 1042, 834, 9269, 3155, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# Fine Tuning

In [22]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [23]:
#setting up device
import torch

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device", device)
model.to(device)

device cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [28]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",
    optim="adamw_torch",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_datasets,
    eval_dataset=val_datasets
)

In [32]:
#train the model
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.409833,0.406614
2,0.401381,0.404721
3,0.391204,0.403760
4,0.383839,0.401474
5,0.378650,0.402446
6,0.374388,0.402339


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.3898825124104818, metrics={'train_runtime': 641.5686, 'train_samples_per_second': 37.408, 'train_steps_per_second': 4.676, 'total_flos': 3248203235328000.0, 'train_loss': 0.3898825124104818, 'epoch': 6.0})

In [35]:
# Move model from TPU (XLA) to CPU
model = model.cpu()

# Save model and tokenizer
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [36]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

# Testing

In [37]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [38]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
